In [15]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd
from numpy import nan

In [16]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine
engine = create_engine('postgresql://zbrothers:%s@localhost:5434/zbrothers' % quote_plus('zbrothers@2026'))

In [17]:
today = datetime.date.today()
# today = datetime.datetime(2026, 4, 15)
today_str = today.strftime('%d/%m/%Y')

In [18]:
statuses = {
        'aberto': {'q': 'aberto', 'qty': 0},
        'enviado': {'q': 'enviado', 'qty': 0},
        'preparando_envio': {'q': 'preparando_envio', 'qty': 0},
        'faturado': {'q': 'faturado', 'qty': 0},
        'pronto_envio': {'q': 'pronto_envio', 'qty': 0},
        'aprovado': {'q': 'aprovado', 'qty': 0},
        'entregue': {'q': 'entregue', 'qty': 0},
        'nao_entregue': {'q': 'nao_entregue', 'qty': 0},
        'cancelado': {'q': 'cancelado', 'qty': 0},
     }

In [19]:
for status in statuses:
        res = requests.get('https://api.tiny.com.br/api2/pedidos.pesquisa.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'situacao': status
                                }
                        )
        
        if res.json()['retorno']['status'] != 'OK':
                print(f"Falha ao obter quantidade de pedidos com status {status}: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
                time.sleep(90) 

                res = requests.get('https://api.tiny.com.br/api2/pedidos.pesquisa.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'situacao': status
                                }
                        )
                
        # Código de erro 20 indica que não há registros nessa consulta
        try:
                statuses[status]['qty'] = res.json()['retorno']['numero_paginas']
        except KeyError:
                if res.json()['retorno']['codigo_erro'] == 20:
                        statuses[status]['qty'] = 0
        except Exception as e:
                print(f"Erro inesperado ao processar status {status}: {e}")
                traceback.print_exc()
                continue

Falha ao obter quantidade de pedidos com status entregue: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...
Falha ao obter quantidade de pedidos com status nao_entregue: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...


In [20]:
all_purchase_orders_all_status = pd.DataFrame()
for status in statuses:
        all_purchase_orders_status = pd.DataFrame()
        for i in range(1, statuses[status]['qty'] + 1):
                print(f"Status: {status}, Quantidade de pedidos: {statuses[status]['qty']}")


                res = requests.get('https://api.tiny.com.br/api2/pedidos.pesquisa.php', 
                                params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                        'formato': 'json',
                                        'situacao': status,
                                        'pagina': i
                                        }
                                )
                
                if res.json()['retorno']['status'] != 'OK':
                        print(f"Falha ao obter pedidos com status {status}: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
                        time.sleep(90) 

                        res = requests.get('https://api.tiny.com.br/api2/pedidos.pesquisa.php', 
                                params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                        'formato': 'json',
                                        'situacao': status,
                                        'pagina': i
                                        }
                                )
                purchase_orders_status = pd.DataFrame([x['pedido'] for x in res.json()['retorno']['pedidos']])
                all_purchase_orders_status = pd.concat([all_purchase_orders_status, purchase_orders_status], ignore_index=True)
        all_purchase_orders_all_status = pd.concat([all_purchase_orders_all_status, all_purchase_orders_status], ignore_index=True)
                        

Status: aberto, Quantidade de pedidos: 3
Status: aberto, Quantidade de pedidos: 3
Status: aberto, Quantidade de pedidos: 3
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, Quantidade de pedidos: 64
Status: enviado, 

In [21]:
all_purchase_orders_all_status

,id,numero,numero_ecommerce,data_pedido,data_prevista,nome,valor,id_vendedor,nome_vendedor,situacao,codigo_rastreamento,url_rastreamento
0,1031446565,8119,NaN,22/09/2023,,BRITANIA ELETRODOMESTICOS S/A,159.60,0,,Em aberto,,
1,1044506894,9758,NaN,10/10/2023,,Extra Cargo Armazéns Gerais LTDA,9000.00,0,,Em aberto,,
2,1053194003,10993,NaN,24/10/2023,,Extra Cargo Armazéns Gerais LTDA,345.40,0,,Em aberto,,
3,1055168023,12343,NaN,07/11/2023,,Extra Cargo Armazéns Gerais LTDA,518.20,0,,Em aberto,,
4,1056888321,18816,NaN,10/01/2024,,ZPORT OPERADORES PORTUARIOS LTDA,4962.40,0,,Em aberto,,
...,...,...,...,...,...,...,...,...,...,...,...,...
30581,1075534068,217379,260427UTMD53CX,26/04/2026,,Thiago soares Loyola,49.99,0,,Cancelado,,
30582,1075534535,217401,260427V1DDTRQT,27/04/2026,,Diomar Gomes De Oliveira,61.15,0,,Cancelado,,
30583,1075534832,217407,2000012695683835,27/04/2026,29/04/2026,Alexandre Gomes de Barros,49.90,0,,Cancelado,768f89fb-9464-59e6-80b1-d,
30584,1075542513,217449,260427VV53QASF,27/04/2026,,Carlos Sergio Bastos Lago,12.99,0,,Cancelado,,


In [22]:
all_purchase_orders_all_status = all_purchase_orders_all_status.replace("", nan)

all_purchase_orders_all_status[['data_pedido_raw', 'data_prevista_raw']] = all_purchase_orders_all_status[['data_pedido', 'data_prevista']].copy()
all_purchase_orders_all_status[['data_pedido', 'data_prevista']] = all_purchase_orders_all_status[['data_pedido', 'data_prevista']].apply(lambda x: pd.to_datetime(x, format='%d/%m/%Y', errors='coerce'))

PURCHASE_ORDER_COLUMNS = ['id', 'numero', 'numero_ecommerce', 'data_pedido', 'data_prevista',
       'nome', 'valor', 'id_vendedor', 'nome_vendedor', 'situacao',
       'codigo_rastreamento', 'url_rastreamento']

In [23]:
all_purchase_orders_all_status

,id,numero,numero_ecommerce,data_pedido,data_prevista,nome,valor,id_vendedor,nome_vendedor,situacao,codigo_rastreamento,url_rastreamento,data_pedido_raw,data_prevista_raw
0,1031446565,8119,NaN,2023-09-22,NaT,BRITANIA ELETRODOMESTICOS S/A,159.60,0,NaN,Em aberto,NaN,NaN,22/09/2023,NaN
1,1044506894,9758,NaN,2023-10-10,NaT,Extra Cargo Armazéns Gerais LTDA,9000.00,0,NaN,Em aberto,NaN,NaN,10/10/2023,NaN
2,1053194003,10993,NaN,2023-10-24,NaT,Extra Cargo Armazéns Gerais LTDA,345.40,0,NaN,Em aberto,NaN,NaN,24/10/2023,NaN
3,1055168023,12343,NaN,2023-11-07,NaT,Extra Cargo Armazéns Gerais LTDA,518.20,0,NaN,Em aberto,NaN,NaN,07/11/2023,NaN
4,1056888321,18816,NaN,2024-01-10,NaT,ZPORT OPERADORES PORTUARIOS LTDA,4962.40,0,NaN,Em aberto,NaN,NaN,10/01/2024,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30581,1075534068,217379,260427UTMD53CX,2026-04-26,NaT,Thiago soares Loyola,49.99,0,NaN,Cancelado,NaN,NaN,26/04/2026,NaN
30582,1075534535,217401,260427V1DDTRQT,2026-04-27,NaT,Diomar Gomes De Oliveira,61.15,0,NaN,Cancelado,NaN,NaN,27/04/2026,NaN
30583,1075534832,217407,2000012695683835,2026-04-27,2026-04-29,Alexandre Gomes de Barros,49.90,0,NaN,Cancelado,768f89fb-9464-59e6-80b1-d,NaN,27/04/2026,29/04/2026
30584,1075542513,217449,260427VV53QASF,2026-04-27,NaT,Carlos Sergio Bastos Lago,12.99,0,NaN,Cancelado,NaN,NaN,27/04/2026,NaN


In [26]:
all_purchase_orders_all_status[PURCHASE_ORDER_COLUMNS].to_sql("purchase_orders", engine, if_exists='append', index=False)

586

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/gustavo/zbrothers/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/home/gustavo/zbrothers/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/gustavo/zbrothers/.venv/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/gustavo/zbrothers/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/home/gustavo/zbrothers/.venv/lib/python3.13/site-packages/i

In [ ]:
df_purchase_orders_details_total = pd.DataFrame()

for id in all_purchase_orders_all_status['id']:
        res = requests.get('https://api.tiny.com.br/api2/pedido.obter.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'id': id
                                }
                        )
        
        if res.json()['retorno']['status'] != 'OK':
                print(f"Falha ao obter detalhes do pedido com ID {id}: {res.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
                time.sleep(90) 

                res = requests.get('https://api.tiny.com.br/api2/pedido.obter.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'id': id
                                }
                        )
                
        try:
                purchase_order_details = pd.DataFrame([res.json()['retorno']['pedido']])
                df_purchase_orders_details_total = pd.concat([df_purchase_orders_details_total, purchase_order_details], ignore_index=True)
        except KeyError:
                print(f"Erro ao processar detalhes do pedido com ID {id}: resposta inesperada da API.")
                traceback.print_exc()
                continue

Falha ao obter detalhes do pedido com ID 926730155: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...
Falha ao obter detalhes do pedido com ID 946551738: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...
Falha ao obter detalhes do pedido com ID 977230274: Erro, possivelmente por sobrecarga na API.
Aguardando 90 segundos para tentar novamente...


In [ ]:
df_purchase_orders_details_total = df_purchase_orders_details_total.replace("", nan)

C:\Users\GustavoWilliamLarsen\AppData\Local\Temp\ipykernel_39904\392140670.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_purchase_orders_details_total = df_purchase_orders_details_total.replace("", nan)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     294 non-null    object 
 1   numero                 294 non-null    object 
 2   numero_ecommerce       242 non-null    object 
 3   data_pedido            294 non-null    object 
 4   data_prevista          230 non-null    object 
 5   data_faturamento       3 non-null      object 
 6   data_envio             0 non-null      float64
 7   data_entrega           0 non-null      float64
 8   id_lista_preco         52 non-null     object 
 9   descricao_lista_preco  294 non-null    object 
 10  cliente                294 non-null    object 
 11  itens                  294 non-null    object 
 12  parcelas               294 non-null    object 
 13  marcadores             294 non-null    object 
 14  condicao_pagamento     8 non-null      object 
 15  forma_